# 10 — Fold3 難関診断

**ベースライン:** 全波数 + RF(SNV+SG1, raw-y) = RMSE 20.19%  
Fold3(樹種15・17・19)の難しさの原因を(a)高含水集中/(b)特定樹種特異性/(c)RF外挿不能/(d)スペクトル飽和に分解する

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

from src.utils import load_data, parse_spectra, get_groups, rmse, setup_japanese_font
from src.preprocessing import snv, savitzky_golay

plt.rcParams['figure.dpi'] = 110
setup_japanese_font()
SEED = 42

# --- Data ---
train_df, _ = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
y = y_s.values.astype(float)
groups = get_groups(train_meta)

# 樹種名辞書 (species_number -> 樹種名)
jp_col = [c for c in train_meta.columns if c not in ['sample number', 'species number']][0]
sp_name = train_meta.drop_duplicates('species number').set_index('species number')[jp_col].to_dict()

# --- CV splits ---
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))
fold_sp = [sorted(set(groups[va])) for _, va in SPLITS]
FOLD3_IDX = 2   # Fold3 = index 2 (species 15,17,19)
tr3, va3 = SPLITS[FOLD3_IDX]
print('Fold validation species:')
for i, sps in enumerate(fold_sp):
    marker = ' <-- FOLD3 (難関)' if i == FOLD3_IDX else ''
    print(f'  Fold{i+1}: {[sp_name.get(s,s) for s in sps]}{marker}')

# --- 前処理 & OOF予測（全波数 RF、ベースライン）---
def preproc(X):
    return savitzky_golay(snv(X), window_length=11, polyorder=2, deriv=1)

def band_mask(center, hw=75):
    return (wn >= center - hw) & (wn <= center + hw)

RF_KW = dict(n_estimators=300, max_features=0.3, random_state=SEED, n_jobs=-1)

print('\nRunning baseline RF OOF predictions (all wavenumbers, SNV+SG1)...')
oof_pred  = np.zeros(len(y))
fold_id   = np.zeros(len(y), dtype=int)  # 1-5
for fi, (tr, va) in enumerate(SPLITS):
    Xtr = preproc(X_raw[tr]); Xva = preproc(X_raw[va])
    rf = RandomForestRegressor(**RF_KW)
    rf.fit(Xtr, y[tr])
    oof_pred[va] = rf.predict(Xva)
    fold_id[va]  = fi + 1

resid = y - oof_pred   # 実測 - 予測  (正=過小予測)
overall_rmse = rmse(y, oof_pred)
fold3_rmse   = rmse(y[va3], oof_pred[va3])
print(f'全体 CV RMSE = {overall_rmse:.2f}%')
print(f'Fold3  RMSE  = {fold3_rmse:.2f}%')

## 診断1: Fold3の中身を可視化する

各foldの樹種・含水率レンジ・件数を確認し、Fold3の構成を把握する。

In [ ]:
# --- 各fold の統計表 ---
rows = []
for fi, (tr, va) in enumerate(SPLITS):
    for sp in fold_sp[fi]:
        sp_mask_va = (groups == sp) & (fold_id == fi + 1)
        ysp = y[sp_mask_va]
        rows.append({'Fold': fi+1,
                     'species': sp,
                     '樹種': sp_name.get(sp,''),
                     'n': len(ysp),
                     'min': round(ysp.min(),1),
                     'max': round(ysp.max(),1),
                     'mean': round(ysp.mean(),1)})

df_folds = pd.DataFrame(rows)
print('=== Fold別 樹種・含水率統計 ===')
print(df_folds.to_string(index=False))

# --- ヒストグラム: 全データ vs Fold3樹種を色分け ---
fold3_sp_list = fold_sp[FOLD3_IDX]
fold3_mask = np.isin(groups, fold3_sp_list)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左: ヒストグラム
axes[0].hist(y[~fold3_mask], bins=40, alpha=0.6, color='steelblue', label='Fold1,2,4,5')
for sp in fold3_sp_list:
    sp_m = groups == sp
    axes[0].hist(y[sp_m], bins=20, alpha=0.7, label=f'Fold3 sp{sp}({sp_name.get(sp,"")})')
axes[0].set_xlabel('含水率 (%)')
axes[0].set_ylabel('件数')
axes[0].set_title('含水率分布: Fold3樹種を色分け')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# 右: 樹種ごとの含水率 box
data_by_sp  = [y[groups == sp] for sp in fold3_sp_list]
labels      = [f'sp{sp}\n{sp_name.get(sp,"")}' for sp in fold3_sp_list]
bp = axes[1].boxplot(data_by_sp, labels=labels, patch_artist=True)
colors_bp = ['#ff7f0e', '#d62728', '#9467bd']
for patch, col in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(col); patch.set_alpha(0.6)
axes[1].set_ylabel('含水率 (%)')
axes[1].set_title('Fold3内の樹種別含水率分布')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/diag1_fold3_contents.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n[Fold3] 含水率 max={y[fold3_mask].max():.1f}%  '
      f'mean={y[fold3_mask].mean():.1f}%  n={fold3_mask.sum()}')
print(f'[他Fold] 含水率 max={y[~fold3_mask].max():.1f}%  '
      f'mean={y[~fold3_mask].mean():.1f}%  n=(~fold3_mask).sum()')

## 診断2: 誤差は「高含水だから」か「樹種が特異だから」か

OOF予測の残差を含水率帯域・樹種別に分解し、バイアスとばらつきを切り分ける。

In [ ]:
# --- 散布図: 予測 vs 実測 ---
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

sp_colors = {15: '#ff7f0e', 17: '#d62728', 19: '#9467bd'}
axes[0].scatter(y[fold_id != 3], oof_pred[fold_id != 3],
                s=6, alpha=0.25, color='gray', label='Fold1,2,4,5')
for sp, col in sp_colors.items():
    m = (fold_id == 3) & (groups == sp)
    axes[0].scatter(y[m], oof_pred[m], s=25, alpha=0.8,
                   color=col, label=f'Fold3 sp{sp} {sp_name.get(sp,"")}')
lim = max(y.max(), oof_pred.max()) * 1.02
axes[0].plot([0, lim], [0, lim], 'r--', lw=1, label='perfect')
axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)
axes[0].set_xlabel('実測含水率 (%)')
axes[0].set_ylabel('予測含水率 (%)')
axes[0].set_title('予測 vs 実測 (全波数RF OOF)')
axes[0].legend(fontsize=7, loc='upper left')
axes[0].grid(True, alpha=0.3)

# --- 残差 box: 含水率帯域別 ---
bands = [(0,50,'0-50%'), (50,100,'50-100%'), (100,200,'100-200%'), (200,400,'200-400%')]
resid_by_band, band_labels = [], []
for lo, hi, lbl in bands:
    m = (y >= lo) & (y < hi)
    resid_by_band.append(resid[m])
    band_labels.append(f'{lbl}\n(n={m.sum()})')
    print(f'{lbl:12s}: mean_resid={resid[m].mean():+.1f}%  RMSE={rmse(y[m],oof_pred[m]):.1f}%')

bp2 = axes[1].boxplot(resid_by_band, labels=band_labels, patch_artist=True,
                       showfliers=True, flierprops=dict(marker='.', ms=3, alpha=0.5))
colors2 = ['#2ca02c','#ffbb78','#ff7f0e','#d62728']
for patch, col in zip(bp2['boxes'], colors2):
    patch.set_facecolor(col); patch.set_alpha(0.6)
axes[1].axhline(0, color='black', lw=1, ls='--')
axes[1].set_ylabel('残差 = 実測 - 予測 (%)')
axes[1].set_title('残差の含水率帯域別分布\n(正=過小予測、負=過大予測)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/diag2_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Fold3: 樹種ごとのバイアス・RMSE ---
print('\n=== Fold3 樹種別 平均残差(バイアス) と RMSE ===')
for sp in fold3_sp_list:
    m = (fold_id == 3) & (groups == sp)
    bias_sp = resid[m].mean()
    rmse_sp = rmse(y[m], oof_pred[m])
    print(f'  sp{sp} {sp_name.get(sp,""):10s}: n={m.sum():3d}  '
          f'mean_resid={bias_sp:+6.1f}%  RMSE={rmse_sp:.1f}%  '
          f'y_max={y[m].max():.1f}%  pred_max={oof_pred[m].max():.1f}%')

## 診断3: RFの「外挿不能」を検証する

RFは訓練データの含水率最大値を超える予測を構造的に出せない。  
Fold3を検証fold にしたとき、学習側最大値 vs 検証側実測最大値を比較する。

In [ ]:
# Fold3の訓練側 vs 検証側の含水率レンジ
y_tr3_max = y[tr3].max()
y_va3_max = y[va3].max()
y_tr3_min = y[tr3].min()

print('=== Fold3: 訓練側 vs 検証側 含水率レンジ ===')
print(f'  訓練側 (Fold1,2,4,5): min={y_tr3_min:.1f}%  max={y_tr3_max:.1f}%  n={len(tr3)}')
print(f'  検証側 (Fold3)      : min={y[va3].min():.1f}%  max={y_va3_max:.1f}%  n={len(va3)}')
print(f'  → 検証側が訓練側上限を {y_va3_max - y_tr3_max:.1f}% 超過している')

# 検証側で訓練上限を超えるサンプル数
exceed_mask = oof_pred[va3] >= (y_tr3_max * 0.95)  # 95%以上 = 頭打ち近辺
exceed_actual_mask = y[va3] > y_tr3_max
print(f'\n  実測が訓練上限({y_tr3_max:.1f}%)超のサンプル: {exceed_actual_mask.sum()}')
print(f'  予測が訓練上限の95%以上に張り付いたサンプル: {exceed_mask.sum()}')

# 散布図: Fold3の予測 vs 実測 with 訓練側上限ライン
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

va3_groups = groups[va3]
for sp, col in sp_colors.items():
    m = va3_groups == sp
    axes[0].scatter(y[va3][m], oof_pred[va3][m], s=25, alpha=0.8,
                   color=col, label=f'sp{sp} {sp_name.get(sp,"")}')
axes[0].axhline(y_tr3_max, color='black', lw=2, ls='--',
               label=f'訓練側上限 {y_tr3_max:.1f}% (RFの予測天井)')
axes[0].axvline(y_tr3_max, color='gray', lw=1, ls=':', alpha=0.7)
lim3 = max(y[va3].max(), oof_pred[va3].max()) * 1.05
axes[0].plot([0, lim3], [0, lim3], 'r--', lw=1)
axes[0].set_xlim(0, lim3); axes[0].set_ylim(0, lim3)
axes[0].set_xlabel('実測含水率 (%)')
axes[0].set_ylabel('予測含水率 (%)')
axes[0].set_title('Fold3: 予測 vs 実測\n(黒破線=訓練側上限=RFの予測天井)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# 残差 vs 実測: 高含水で系統的な過小予測？
resid_va3 = resid[va3]
for sp, col in sp_colors.items():
    m = va3_groups == sp
    axes[1].scatter(y[va3][m], resid_va3[m], s=25, alpha=0.8,
                   color=col, label=f'sp{sp} {sp_name.get(sp,"")}')
axes[1].axhline(0, color='black', lw=1, ls='--')
axes[1].axvline(y_tr3_max, color='black', lw=2, ls='--', alpha=0.7,
               label=f'訓練上限 {y_tr3_max:.1f}%')
axes[1].set_xlabel('実測含水率 (%)')
axes[1].set_ylabel('残差 = 実測 - 予測 (%)')
axes[1].set_title('Fold3: 残差 vs 実測含水率\n(正=過小予測)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/diag3_rf_extrapolation.png', dpi=150, bbox_inches='tight')
plt.show()

# 定量: 実測>訓練上限のサンプルの平均残差
high_mc_mask_va3 = y[va3] > y_tr3_max
if high_mc_mask_va3.sum() > 0:
    mean_resid_high = resid[va3][high_mc_mask_va3].mean()
    print(f'\n実測 > 訓練上限({y_tr3_max:.1f}%) のサンプル: n={high_mc_mask_va3.sum()}')
    print(f'  平均残差 = {mean_resid_high:+.1f}% ({"過小予測" if mean_resid_high > 0 else "過大予測"})')
    print(f'  RMSE     = {rmse(y[va3][high_mc_mask_va3], oof_pred[va3][high_mc_mask_va3]):.1f}%')
else:
    print('実測が訓練上限を超えるサンプルなし')

## 診断4: 高含水域でスペクトル情報が飽和していないか

汎化核である4760・6900帯の吸収値を含水率に対してプロットし、飽和の有無を確認する。

In [ ]:
X_pp = preproc(X_raw)

abs_4760 = X_pp[:, band_mask(4760)].mean(1)
abs_6900 = X_pp[:, band_mask(6900)].mean(1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, feat, bname in zip(
        axes[:2], [abs_4760, abs_6900], ['4760', '6900']):
    # 他fold (gray)
    ax.scatter(y[~fold3_mask], feat[~fold3_mask],
               s=5, alpha=0.2, color='gray', label='Fold1,2,4,5')
    for sp, col in sp_colors.items():
        m = groups == sp
        ax.scatter(y[m], feat[m], s=20, alpha=0.7,
                   color=col, label=f'sp{sp} {sp_name.get(sp,"")}')
    ax.set_xlabel('含水率 (%)')
    ax.set_ylabel(f'{bname} cm^-1 SNV+SG1値')
    ax.set_title(f'{bname} cm^-1 vs 含水率\n(飽和すれば高含水域で水平になる)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# 含水率帯域別に4760帯の分散を比較
bands_mc = [(0,50),(50,100),(100,200),(200,400)]
var_by_band, n_by_band, lbl_by_band = [], [], []
print('=== 4760帯吸収値の帯域別分散 (飽和すれば分散が縮小) ===')
for lo, hi in bands_mc:
    m = (y >= lo) & (y < hi)
    if m.sum() < 3: continue
    v = abs_4760[m].std()
    r_val = np.corrcoef(y[m], abs_4760[m])[0,1]
    var_by_band.append(v); n_by_band.append(m.sum())
    lbl = f'{lo}-{hi}%\n(n={m.sum()})'
    lbl_by_band.append(lbl)
    print(f'  {lo:3d}-{hi:3d}%: std={v:.4f}  r={r_val:+.3f}  n={m.sum()}')

axes[2].bar(lbl_by_band, var_by_band, color=['#2ca02c','#ffbb78','#ff7f0e','#d62728'])
axes[2].set_ylabel('4760帯吸収値の std')
axes[2].set_title('4760帯のばらつき（帯域別）\n(飽和すると高含水でstd低下)')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/diag4_spectral_saturation.png', dpi=150, bbox_inches='tight')
plt.show()

# Fold3樹種 vs 他での4760帯の分布を確認
print(f'\n4760帯吸収値:')
print(f'  Fold3樹種 (sp15,17,19): mean={abs_4760[fold3_mask].mean():.4f}  '
      f'std={abs_4760[fold3_mask].std():.4f}')
print(f'  他の樹種             : mean={abs_4760[~fold3_mask].mean():.4f}  '
      f'std={abs_4760[~fold3_mask].std():.4f}')

## 診断5: モデル別にFold3と高含水の振る舞いを比較

全波数のまま、外挿できるモデル(Ridge)と木ベースモデル(RF, HistGB)を比較。  
Fold3のRMSEと高含水サンプル(>y_tr3_max)の平均残差に注目する。

In [ ]:
def run_fold3_only(model_fn, preproc_fn=preproc, scaler=False):
    """Fold3だけ検証し、RMSE・高含水バイアスを返す。"""
    Xtr = preproc_fn(X_raw[tr3]); Xva = preproc_fn(X_raw[va3])
    if scaler:
        sc = StandardScaler()
        Xtr = sc.fit_transform(Xtr); Xva = sc.transform(Xva)
    m = model_fn()
    m.fit(Xtr, y[tr3])
    pred = m.predict(Xva)
    if hasattr(pred, 'ravel'): pred = pred.ravel()
    fold3_r  = rmse(y[va3], pred)
    resid_va = y[va3] - pred
    hi_mask  = y[va3] > y_tr3_max   # 訓練上限を超えるサンプル
    hi_bias  = resid_va[hi_mask].mean() if hi_mask.sum() > 0 else np.nan
    hi_rmse  = rmse(y[va3][hi_mask], pred[hi_mask]) if hi_mask.sum() > 0 else np.nan
    pred_max = pred.max()
    return fold3_r, hi_bias, hi_rmse, pred_max, pred

models_d5 = [
    ('RF',    lambda: RandomForestRegressor(**RF_KW),     False),
    ('HistGB',lambda: HistGradientBoostingRegressor(
                          max_iter=300, random_state=SEED), False),
    ('Ridge', lambda: Ridge(alpha=100),                   True),
]

print(f'訓練側上限: {y_tr3_max:.1f}%   実測最大: {y_va3_max:.1f}%')
print(f'{"モデル":<10} {"Fold3 RMSE":>12} {"超過平均残差":>14} {"超過RMSE":>10} {"予測最大値":>10}')
print('-' * 60)

all_preds_d5 = {}
for name, fn, sc in models_d5:
    f3r, hbias, hrmse, pmax, pva = run_fold3_only(fn, scaler=sc)
    all_preds_d5[name] = pva
    hbias_s = f'{hbias:+.1f}%' if not np.isnan(hbias) else 'N/A'
    hrmse_s = f'{hrmse:.1f}%'  if not np.isnan(hrmse) else 'N/A'
    print(f'{name:<10} {f3r:>12.2f}% {hbias_s:>14} {hrmse_s:>10} {pmax:>10.1f}%')

# 散布図: 3モデルの Fold3 予測 vs 実測
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, fn, sc) in zip(axes, models_d5):
    pva = all_preds_d5[name]
    for sp, col in sp_colors.items():
        m = groups[va3] == sp
        ax.scatter(y[va3][m], pva[m], s=20, alpha=0.7,
                   color=col, label=f'sp{sp}')
    ax.axhline(y_tr3_max, color='black', lw=2, ls='--', alpha=0.7,
               label=f'訓練上限 {y_tr3_max:.1f}%')
    lm = max(y[va3].max(), pva.max()) * 1.05
    ax.plot([0, lm], [0, lm], 'r--', lw=1)
    ax.set_xlim(0, lm); ax.set_ylim(0, lm)
    ax.set_xlabel('実測含水率 (%)')
    ax.set_ylabel('予測含水率 (%)')
    ax.set_title(f'{name}: Fold3 予測 vs 実測\nFold3 RMSE={rmse(y[va3],pva):.1f}%')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/diag5_model_fold3.png', dpi=150, bbox_inches='tight')
plt.show()

## 総括（実行後に記入）

In [ ]:
# =======================================================================
# 総括サマリー数値
# =======================================================================
print('=' * 65)
print('FOLD3 診断サマリー')
print('=' * 65)

print(f'\n[Fold3 構成]')
for sp in fold3_sp_list:
    m = groups == sp
    print(f'  sp{sp} {sp_name.get(sp,""):12s}: '
          f'n={m.sum()}  y={y[m].min():.0f}-{y[m].max():.0f}%  mean={y[m].mean():.0f}%')

print(f'\n[外挿不能の規模]')
exceed = (y[va3] > y_tr3_max).sum()
print(f'  訓練側上限: {y_tr3_max:.1f}%  Fold3検証上限: {y_va3_max:.1f}%')
print(f'  検証側で訓練上限を超えるサンプル: {exceed}/{len(va3)} ({100*exceed/len(va3):.1f}%)')

print(f'\n[帯域別バイアス (残差 = 実測-予測)]')
for lo, hi, lbl in bands:
    m = (y >= lo) & (y < hi)
    if m.sum() > 0:
        b = resid[m].mean()
        print(f'  {lbl:12s}: mean_resid={b:+.1f}%  ({"過小" if b>0 else "過大"}予測)')

print(f'\n[Fold3 樹種別]')
for sp in fold3_sp_list:
    m = (fold_id == 3) & (groups == sp)
    b = resid[m].mean()
    r = rmse(y[m], oof_pred[m])
    print(f'  sp{sp} {sp_name.get(sp,""):12s}: '
          f'bias={b:+.1f}%  RMSE={r:.1f}%  y_max={y[m].max():.0f}%')

print()
print('難しさの原因 (a)-(d) に対する暫定評価:')
print('  -> 実行後に図・数値を見て記入')